In [ ]:
import glob, json
import numpy as np
import matplotlib.pyplot as plt
import causal_effect as ce

# validated categorical slots 1-3 (light surface)
S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#a8a7a1"
SURFACE = "#ffffff"
SERIES = [S1, S2, S3]

def load_all(x, y="TMS_nox_clean", root="effects"):
    """해당 원인-결과 쌍의 모든 해상도 결과 (하위 폴더 포함), 구간 크기 순."""
    pat = f"{root}/**/*__{ce._slug(x)}__{ce._slug(y)}.npz"
    files = glob.glob(pat, recursive=True) + glob.glob(
        f"{root}/*__{ce._slug(x)}__{ce._slug(y)}.npz")
    out = sorted((ce.load_effect(f) for f in sorted(set(files))),
                 key=lambda d: d["meta"]["minutes_per_lag"])
    if not out:
        raise FileNotFoundError(f"결과 파일 없음: {pat}")
    return out



def ci(d, conf=None):
    """저장된 부트스트랩 표본에서 임의 신뢰수준의 구간을 다시 계산."""
    if conf is None or d["draws"].size == 0:
        return d["lo"], d["hi"]          # 예전 파일: 저장된 수준 그대로
    q = 100 * (1 - conf) / 2
    return np.percentile(d["draws"], [q, 100 - q], axis=0)

def sig(d, conf=None):
    lo, hi = ci(d, conf)
    return np.isfinite(lo) & np.isfinite(hi) & (lo * hi > 0)


def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(MUTED); ax.spines[s].set_linewidth(0.8)
    ax.tick_params(colors=INK2, labelsize=9, length=3, width=0.8)
    ax.grid(axis="y", color=MUTED, alpha=0.35, linewidth=0.6)
    ax.set_axisbelow(True)

runs = load_all("ratio_2")          # ← 변수명을 넘겨야 함
for d in runs:
    print(d["meta"]["freq"], "lags", d["lags"][-1], "boot", d["meta"]["boot"])



In [ ]:
import matplotlib
from pathlib import Path
from matplotlib import font_manager

# ===================== 설정 =====================
XVAR = "ratio_main"          # 원인 변수
YVAR = "TMS_nox_clean"       # 결과 변수
BINS  = [5, 10, 15]          # None = 전부, 또는 [5, 10, 15] (분 단위)
BANDS = None                 # None = 전부 음영, 또는 [10] 처럼 일부만
CONF_PLOT = None             # None = 저장된 수준, 또는 0.95 / 0.90
ANCHOR = "end"               # 각 값을 찍을 위치: "end" 구간 끝 / "center" 가운데 / "start" 시작
TITLE = None                 # None = 변수명에서 자동 생성
NOTE  = None                 # 부제에 덧붙일 문구
FIGSIZE = (14, 8)
SAVE = True

# 구간 크기별 고정 색. 어떤 해상도를 빼도 나머지 색이 바뀌지 않도록 크기로 키를 잡음
BIN_COLORS = {3: "#4a3aa7", 5: "#2a78d6", 10: "#eb6834",
              15: "#1baf7a", 20: "#e87ba4", 30: "#e34948"}

NAMES = {                    # 변수명 -> 표시 이름. 없으면 변수명 그대로
    "TMS_nox_clean": "NOx",
    "ratio_main": "전체 산소/오일 비율", "oil_main": "오일 투입량",
    "ratio_1": "버너 1 산소/오일 비율", "ratio_2": "버너 2 산소/오일 비율",
    "ratio_3": "버너 3 산소/오일 비율", "ratio_4": "버너 4 산소/오일 비율",
    "oil_1": "1번 오일 투입량", "oil_2": "2번 오일 투입량",
    "oil_3": "3번 오일 투입량", "oil_4": "4번 오일 투입량",
    "oxy_1": "1번 산소 투입량", "oxy_2": "2번 산소 투입량",
    "oxy_3": "3번 산소 투입량", "oxy_4": "4번 산소 투입량",
    "pull": "용출량", "ARCH #1": "천장 온도 #1",
    "arch_3_4": "천장 온도(3·4 평균)", "bt_temp": "바닥 온도(평균)",
    "WEATHER_temp": "외기 온도", "WEATHER_humi": "외기 습도",
    "WEATHER_acc_precip": "누적 강수량",
}
ANCHOR_TXT = {"end": "각 구간의 끝", "center": "각 구간의 가운데",
              "start": "각 구간의 시작"}
# ===============================================

def label(v):
    return NAMES.get(v, v)


def xpos(d):
    """각 시차 값을 몇 분에 찍을지.

    값은 구간 [k*b, (k+1)*b) 전체에 걸친 변화량이므로, 기본은 구간 끝에 찍는다
    (예: 10분 간격의 시차 1은 10~20분을 덮으므로 20분에 표시).
    """
    b   = d["meta"]["minutes_per_lag"]
    off = {"end": b, "center": b / 2, "start": 0}[ANCHOR]
    return d["minutes"] + off


# 한글 폰트 -- 없으면 네모로 표시됨
have = {f.name for f in font_manager.fontManager.ttflist}
for cand in ("Malgun Gothic", "NanumGothic", "NanumBarunGothic", "AppleGothic"):
    if cand in have:
        matplotlib.rcParams["font.family"] = cand
        break
else:
    print("한글 폰트를 찾지 못했습니다 -- 'Malgun Gothic' 설치 필요")
matplotlib.rcParams["axes.unicode_minus"] = False

runs = load_all(XVAR, YVAR)
if BINS is not None:
    runs = [d for d in runs if d["meta"]["minutes_per_lag"] in BINS]
if not runs:
    raise SystemExit("선택한 해상도의 결과가 없습니다.")

unknown = [d["meta"]["minutes_per_lag"] for d in runs
           if d["meta"]["minutes_per_lag"] not in BIN_COLORS]
if unknown:
    print(f"BIN_COLORS에 없는 구간 크기 {sorted(set(unknown))} -- 회색으로 표시됩니다")
if len(runs) > 3:
    print(f"주의: {len(runs)}개 해상도 -- 색 구분이 어려워집니다. BINS로 줄이는 편이 낫습니다")

mt   = runs[0]["meta"]
unit = f"+{mt['step_pct']:g}%" if mt["islog"] else "+1 단위"   # 로그 변수만 % 해석
conf = CONF_PLOT if CONF_PLOT is not None else mt["conf"]

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=110, facecolor=SURFACE)
style(ax)

for d in runs:
    b = d["meta"]["minutes_per_lag"]
    c = BIN_COLORS.get(b, MUTED)
    m, psi = xpos(d), d["psi"]
    lo, hi = ci(d, CONF_PLOT)
    keep = d["status"] != "not_identifiable"     # 추정 불가 구간은 선을 끊음

    if BANDS is None or b in BANDS:
        ax.fill_between(m, np.where(keep & np.isfinite(lo), lo, np.nan),
                        np.where(keep & np.isfinite(hi), hi, np.nan),
                        color=c, alpha=0.15, linewidth=0)
    ax.plot(m, np.where(keep, psi, np.nan), "-o", color=c, linewidth=2.5,
            markersize=7, label=f"{b}분 간격")

ax.axhline(0, color=INK2, linewidth=1.2)

for d in runs:                                   # 최댓값만 표시
    if not np.isfinite(d["psi"]).any():
        continue
    k = int(np.nanargmax(d["psi"]))
    ax.annotate(f"{d['psi'][k]:+.1f}", (xpos(d)[k], d["psi"][k]),
                textcoords="offset points", xytext=(8, 10),
                fontsize=15, color=INK, fontweight="bold")

ax.set_xlim(-3, max(xpos(r)[-1] for r in runs) + 4)
ax.xaxis.set_major_locator(matplotlib.ticker.MultipleLocator(5))

ax.set_xlabel(f"스텝 이후 경과 시간 (분, {ANCHOR_TXT[ANCHOR]})",
              fontsize=15, color=INK2, labelpad=10)
ax.set_ylabel(f"{label(YVAR)} 구간별 변화량 ({label(XVAR)} {unit} 당)",
              fontsize=15, color=INK2, labelpad=10)
ax.tick_params(labelsize=13)
ax.legend(frameon=False, fontsize=14, labelcolor=INK2,
          title="샘플링 주기", title_fontsize=14, loc="upper right")

ax.set_title(TITLE or f"{label(XVAR)} {unit} 상승 시 {label(YVAR)} 반응",
             fontsize=20, color=INK, loc="left", fontweight="bold", pad=30)
sub = f"음영 = {conf:.0%} 부트스트랩 신뢰구간"
ax.annotate(sub + (f" · {NOTE}" if NOTE else ""),
            xy=(0, 1.02), xycoords="axes fraction", fontsize=13, color=INK2)

fig.tight_layout()
if SAVE:
    out = Path("plots/effects"); out.mkdir(parents=True, exist_ok=True)
    bintag = "all" if BINS is None else "-".join(str(b) for b in sorted(BINS))
    fig.savefig(out / f"line_{XVAR}_{YVAR}_{bintag}.png", dpi=200,
                facecolor=SURFACE)
plt.show()


In [ ]:
# ===================== 설정 =====================
TARGET_MIN = 20               # 비교할 경과 시간 (분). 각 해상도에서 가장 가까운 구간 끝을 사용
VARS  = ["ratio_1", "ratio_2", "ratio_3"]
YV    = "TMS_nox_clean"
BINS  = [5, 10, 15]           # None = 전부, 또는 [5, 10] 처럼 지정 (분 단위)
CONF_PLOT = None              # None = 저장된 수준
SAVE  = True
# ===============================================

# 구간 크기별 고정 색. 어떤 해상도를 빼도 나머지 색이 바뀌지 않도록 크기로 키를 잡음
BIN_COLORS = {3: "#4a3aa7", 5: SERIES[0], 10: SERIES[1], 15: SERIES[2],
              20: "#e87ba4", 30: "#e34948"}

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def cum_at(d, minutes, conf=None):
    """`minutes` 시점까지의 누적 반응. 구간 끝이 minutes에 가장 가까운 시차까지 합산.

    누적값은 벽시계 시각 기준이라 해상도가 달라도 같은 뜻을 갖는다.
    신뢰구간은 각 부트스트랩 표본을 먼저 누적한 뒤 분위수를 잡아야 한다.
    """
    b   = int(d["meta"]["minutes_per_lag"])
    end = (d["lags"] + 1) * b                      # 각 시차가 덮는 구간의 끝
    k   = int(np.argmin(np.abs(end - minutes)))    # 가장 가까운 구간 끝
    st  = d["status"][: k + 1]
    has_ok  = (st == "ok").any()
    has_nid = (st == "not_identifiable").any()

    if (st == "no_causal_path").all():
        status = "no_causal_path"              # 그래프상 경로 자체가 없음
    elif not has_ok:
        status = "not_identifiable"            # 추정된 구간이 하나도 없음
    elif has_nid:
        status = "lower_bound"                 # 일부 구간을 0으로 세고 넘어감
    else:
        status = "ok"


    val = float(np.nancumsum(d["psi"][: k + 1])[-1])
    dr  = d["draws"]
    if dr.size == 0 or status in ("no_causal_path", "not_identifiable"):
        lo = hi = np.nan
    else:
        c = np.nancumsum(dr[:, : k + 1], axis=1)[:, -1]
        q = 100 * (1 - (conf if conf is not None else d["meta"]["conf"])) / 2
        lo, hi = np.percentile(c, [q, 100 - q])
    return dict(mins=b, lag=int(d["lags"][k]), elapsed=int(end[k]),
                psi=val, lo=float(lo), hi=float(hi), status=status,
                n_skip=int((st == "not_identifiable").sum()))


def fmt(v):
    """작은 값이 '0'으로 보이지 않도록 -- 구조적 0과 헷갈림."""
    return f"{v:+.2f}" if abs(v) < 0.5 else f"{v:+.1f}"


# 변수별로 모든 해상도 결과 수집 (없는 변수는 건너뜀)
data, missing = {}, []
for v in VARS:
    try:
        rows = [cum_at(d, TARGET_MIN, CONF_PLOT) for d in load_all(v, YV)]
    except FileNotFoundError:
        missing.append(v); continue
    if BINS is not None:
        rows = [r for r in rows if r["mins"] in BINS]
    (data.setdefault(v, rows) if rows else missing.append(v))
if missing:
    print("결과 없음 (또는 선택한 해상도 없음):", ", ".join(missing))
if not data:
    raise SystemExit("그릴 결과가 없습니다.")

bins  = sorted({r["mins"] for rows in data.values() for r in rows})
CBIN  = {b: BIN_COLORS.get(b, MUTED) for b in bins}
names = list(data)
if len(bins) > 3:
    print(f"주의: {len(bins)}개 해상도를 한 번에 표시 -- 색 구분이 어려워집니다")

# 해상도별로 실제 합산이 끝난 시각 -- TARGET_MIN과 다를 수 있다
used = {r["mins"]: r["elapsed"] for rows in data.values() for r in rows}
for b, e in sorted(used.items()):
    flag = "" if e == TARGET_MIN else "  <-- 목표 시각과 다름"
    print(f"  {b}분 간격 → 0~{e}분 누적{flag}")

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=110,
                       facecolor=SURFACE)
style(ax)

W  = 0.66
bw = W / max(len(bins), 1) * 0.86
lab_kw = dict(ha="center", fontsize=12, fontweight="bold", zorder=5)
n_lb = 0

for j, v in enumerate(names):
    for r in data[v]:
        i  = bins.index(r["mins"])
        dx = (i - (len(bins) - 1) / 2) * (W / max(len(bins), 1))
        x, c = j + dx, CBIN[r["mins"]]

        if r["status"] in ("ok", "lower_bound"):
            # 신뢰구간이 0을 포함하면 회색 -- 0과 구별되지 않는 값에 색을 주지 않음
            hit = (np.isfinite(r["lo"]) and np.isfinite(r["hi"])
                   and r["lo"] * r["hi"] > 0)
            lb  = r["status"] == "lower_bound"
            n_lb += lb
            ax.bar(x, r["psi"], width=bw, color=c if hit else "#d6d5d0",
                   edgecolor=c if lb else "none", linewidth=1.6,
                   hatch="///" if lb else None, alpha=0.55 if lb else 1.0,
                   zorder=2)
            ax.errorbar(x, r["psi"],
                        yerr=[[r["psi"] - r["lo"]], [r["hi"] - r["psi"]]],
                        fmt="none", ecolor=INK2, elinewidth=1.4, capsize=4,
                        zorder=3)
            top = r["hi"] if r["psi"] >= 0 else r["lo"]
            if not np.isfinite(top):
                top = r["psi"]
            ax.annotate(("≥ " if lb else "") + fmt(r["psi"]), (x, top),
                        textcoords="offset points",
                        xytext=(0, 7 if r["psi"] >= 0 else -18),
                        color=INK if hit else INK2, **lab_kw)

        elif r["status"] == "no_causal_path":
            # 측정된 0이 아니라, 그래프에 경로 자체가 없다는 뜻
            ax.plot([x - bw / 2, x + bw / 2], [0, 0], color=c, linewidth=6,
                    solid_capstyle="butt", zorder=5)
            ax.annotate("경로\n없음", (x, 0), textcoords="offset points",
                        xytext=(0, 9), ha="center", fontsize=10,
                        color=INK2, zorder=5)

        else:                               # not_identifiable
            ax.plot([x], [0], "x", color=MUTED, markersize=13,
                    markeredgewidth=2.6, zorder=4)
            ax.annotate("식별\n불가", (x, 0), textcoords="offset points",
                        xytext=(0, 9), ha="center", fontsize=10,
                        color=MUTED, zorder=5)

ax.axhline(0, color=INK2, linewidth=1.2, zorder=1)   # 0선은 막대 아래로

vals = [x for rows in data.values() for r in rows
        for x in (r["lo"], r["hi"], r["psi"]) if np.isfinite(x)]
lo_y, hi_y = min(min(vals), 0.0), max(max(vals), 0.0)
pad = 0.20 * ((hi_y - lo_y) or 1.0)
ax.set_ylim(lo_y - pad, hi_y + pad)

ax.set_xticks(range(len(names)))
ax.set_xticklabels([label(v) for v in names], fontsize=14)
ax.set_xlim(-0.6, len(names) - 0.4)
if "ratio_main" in names and len(names) > 1:
    ax.axvline(names.index("ratio_main") + 0.5, color=MUTED,
               linewidth=1.0, linestyle=":", zorder=0)
ax.grid(axis="x", visible=False)
ax.tick_params(axis="y", labelsize=13)
ax.set_ylabel(f"{label(YV)} 누적 변화량 (+1% 당)", fontsize=15,
              color=INK2, labelpad=10)

handles = [Patch(facecolor=CBIN[b], edgecolor="none",
                 label=f"{b}분 간격 (0~{used[b]}분 누적)") for b in bins]
handles += [Patch(facecolor="#d6d5d0", edgecolor="none",
                  label="0과 구별되지 않음"),
            Line2D([], [], color=INK2, linewidth=4,
                   label="경로 없음 (그래프상 효과 0)"),
            Line2D([], [], color=MUTED, marker="x", linestyle="none",
                   markersize=10, markeredgewidth=2.4,
                   label="식별 불가 (추정 불가능)")]
if n_lb:
    handles.append(Patch(facecolor="none", edgecolor=INK2, hatch="///",
                         label="하한 (일부 구간 식별 불가)"))

def row_major(h, ncol):
    """matplotlib은 열 우선으로 채우므로, 가로로 읽히도록 순서를 뒤집는다."""
    nrow = -(-len(h) // ncol)
    return [h[r * ncol + c] for c in range(ncol) for r in range(nrow)
            if r * ncol + c < len(h)]

NCOL = 4
ax.legend(handles=row_major(handles, NCOL), frameon=False, fontsize=13,
          labelcolor=INK2, loc="upper left", bbox_to_anchor=(0, -0.12),
          ncol=NCOL, columnspacing=2.0, handlelength=1.6,
          handletextpad=0.6, borderaxespad=0)

ax.set_title(f"{TARGET_MIN}분까지 누적된 {label(YV)} 반응",
             fontsize=19, color=INK, loc="left", fontweight="bold", pad=30)
ax.annotate("오차막대 = 부트스트랩 신뢰구간 · 누적값은 벽시계 시각 기준이라 "
            "해상도가 달라도 직접 비교할 수 있음",
            xy=(0, 1.02), xycoords="axes fraction", fontsize=12.5, color=INK2)

fig.tight_layout(rect=(0, 0.10, 1, 1))    # 축 밖 범례 자리 확보
if SAVE:
    out = Path("plots/effects"); out.mkdir(parents=True, exist_ok=True)
    bintag = "all" if BINS is None else "-".join(str(b) for b in sorted(BINS))
    fig.savefig(out / f"bar_cum_{YV}_{TARGET_MIN}_{bintag}.png", dpi=200,
                facecolor=SURFACE, bbox_inches="tight")
plt.show()
